In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join('.'))
if module_path not in sys.path:
    sys.path.append(module_path + "/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline

import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib as mpl
import matplotlib.pyplot as plt
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# Observational data (OIPC, ETOPO5, IMERG). Override with OBS_DATA_DIR; see config/paths.env.example
obspath = os.environ.get('OBS_DATA_DIR', 'data/external')
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR')
if not opath:
    raise RuntimeError(
        "FIG_OUTPUT_DIR is not set. Run `source config/paths.env` before launching Jupyter "
        "(see config/paths.env.example). Refusing to guess a default: the old fallback wrote to "
        "a repo-local outputs/ that tools/sync_manuscript_figs.sh never read, and figures "
        "silently diverged between the two directories."
    )
os.makedirs(opath, exist_ok=True)

**read in data**
---
Precipitation isotope data: Online Isotopes in Precipitation Calculator<br>
source: https://wateriso.utah.edu/waterisotopes/index.html <br>
Bowen, G. J. (Year) The Online Isotopes in Precipitation Calculator, version X.X. http://www.waterisotopes.org

Precipitation rate data: Integrated multi-satellite retrievals for the global precipitation measurement (GPM) mission (IMERG)<br>
source: https://gpm.nasa.gov/data/imerg <br>
reference: doi.org/10.1007/978-3-030-24568-9_19

Topography: National Geophysical Data Center 5-minute Gridded Global Relief Data (ETOPO5)<br>
source: https://www.ncei.noaa.gov/products/etopo-global-relief-model <br>
reference: doi.org/10.7289/V5D798BF


In [ ]:
# lat/lon subsetting bounds
lon_min, lon_max = -125, -85
lat_min, lat_max = 10, 42

# --- ETOPO05 topography --- #
filen=f'{obspath}/obs.etopo5.zsurf.nc'
etopo_full=xr.open_dataset(f'{filen}').ROSE / 1000 # convert from m to km
etopo_full.attrs['units'] = 'km'
etopo_full=longitude_flip(etopo_full)
etopo_land=etopo_full.where(etopo_full>0, np.nan) # mask bathymetry
etopo_land = etopo_land.rename({'ETOPO05_Y':'lat','ETOPO05_X':'lon'})

# --- OIPC isotopes --- #
filen='OIPC_monthly_data.nc'
oipc=xr.open_dataset(f'{obspath}/{filen}').isotopes[::-1,:,:].rename({'Lat':'lat','Lon':'lon'})
# OIPC grid registration correction 
# The file's coordinates place the field about 0.5 deg east and 0.25 deg south of where it
# belongs. Plotted as-is the data sits southeast of the coastline -- most visible as a white gap
# down the west side of Baja with colour spilling into the Gulf.
#
# Measured, not guessed: OIPC is land-only (NaN over ocean), so its own NaN pattern traces a
# coastline. Fitting that against the Natural Earth 50m polygons cartopy actually draws raises
# coastal-cell agreement over this domain from 0.82 to 0.96, and the optimum is dlon=-0.50,
# dlat=+0.25 -- exactly 6 and 3 cells of the 1/12 deg grid. Fitting the Gulf of Mexico, the US
# Pacific Northwest, northern South America and the western Mediterranean independently recovers
# the same shift to within the fit's resolution, so this is a global property of the file, not a
# regional or projection artefact.
#
# This is an EMPIRICAL registration, not a documented correction, and the root cause in the
# source file is unknown. If a corrected OIPC file turns up, delete this and use it instead.
# Applied before subsetting so lon_min/lat_min above mean what they say.
OIPC_DLON, OIPC_DLAT = -0.50, 0.25
oipc=oipc.assign_coords(lon=oipc.lon + OIPC_DLON, lat=oipc.lat + OIPC_DLAT)

oipc=oipc.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max))
oipc=oipc.transpose('month','lat','lon')

# --- IMERG precipitation rate --- #
"""
filen='/Users/dervlamk/OneDrive/data/obs/satellite/imerg/imerg_V07_19980101-20241231_monthly_gn.nc'
ds=xr.open_dataset(f'{filen}').precipitation * 24 # convert from mm/hr to mm/day
ds=ds.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).groupby('time.month').mean(dim='time') # calculate monthly climatologies
del ds.attrs["units"]
imerg.attrs['Units'] = 'mm/day'
"""
filen='/Users/dervlamk/OneDrive/data/obs/satellite/imerg/imerg.gn.timeseries.2001-2018.nc'
ds=xr.open_dataset(filen).precipitation  * 24 # convert from mm/hr to mm/day
ds=ds.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).groupby('time.month').mean(dim='time') # calculate monthly climatologies
del ds.attrs["units"]
ds.attrs['Units'] = 'mm/day'
imerg=ds.transpose('month','lat','lon')


In [ ]:
# calculate seasonal dD and prec values (assumes OIPC data is already flux-weighted)

# init dicitionaries
dD = {}
prec = {}
pprec = {}

seasons=np.array(['djf','jfm','mam','jja','jas','jjas','son','ann'])

# sum annual total precip
annual_total_p = imerg.sum(dim="month")

# get seasonal means of obs data
for season in seasons:
    months = get_season(season)
    dD[season] = oipc.isel(month=months).mean(dim='month') # isotopes
    prec[season] = imerg.isel(month=months).mean(dim='month') # precip
    
    if season != 'ann':
        pprec[season] = imerg.isel(month=months).sum(dim='month') / annual_total_p
    else:
        pprec[season] = imerg / annual_total_p
        
    pprec[season].attrs['Units'] = '%'
    pprec[season].attrs['newname'] = 'percent of annual total'

In [ ]:
cmap_low = cmo.speed_r
cmap_up = cmo.turbid 
ccmap = combine_cmaps_white_center(cmap_low, cmap_up, range_low=[0,.8], range_up=[.1,.9], n_low=128, n_up=128, n_white=11)
#ccmap = combine_cmaps(cmap_low, cmap_up, range_low=[0,.825], range_up=[.15,1], n_low=128, n_up=128)
ccmap

In [ ]:
# Core locations
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
cnames=np.array(['NH22P','DSDP\n480/479'])
# Model Data
olon = dD['ann'].lon; olat = dD['ann'].lat
plon = prec['ann'].lon; plat = prec['ann'].lat
# settings
lw=1
arrow_kw={'arrowstyle':'-|>', 'color':'k', 'linewidth':1.25, 'shrinkB':6}
text_kw={'fontsize':12, 'fontweight':'normal', 'ha':'center'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':16, 'ha':'center', 'va':'center'}
titles=np.array([u'Summer$-$Winter δD$_{\mathbf{prec}}$', 'Fraction of annual precipitation falling in Summer'])
letters=np.array(['a','b'])
# NAM domain outline (single merged polygon, see scripts/py_functions/domain_funcs.py)
nam_domain=nam_domain_outline()
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -85., 10., 37.]
tx=np.abs((map_bnds[0]-map_bnds[1])/2) + map_bnds[0]
ty=map_bnds[3]+0.5
# isotopes cmap
icmap=ccmap #cmo.balance
ivmin=-60
ivmax=60
ilevels=np.linspace(ivmin, ivmax, 41)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
pcmap=cmo.rain
pvmin=10
pvmax=90
plevels=np.linspace(pvmin, pvmax, 41)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,'', **text_kw)

#dDp
dDdiff = dD['jas'] - dD['jfm']
cf1=ax[0].pcolormesh(olon, olat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0].annotate(cnames[1], xy=(clons[1],clats[1]), xytext=(clons[1]-5,clats[1]-6),
               arrowprops=arrow_kw, **text_kw)
ax[0].annotate(cnames[0], xy=(clons[0],clats[0]), xytext=(clons[0]-3,clats[0]-5.5),
               arrowprops=arrow_kw, **text_kw)
ax[0].add_feature(cfeature.OCEAN, fc='lightgrey')

# precip
cf2=ax[1].pcolormesh(plon, plat, pprec['jas']*100, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].contour(etopo_land.lon, etopo_land.lat, etopo_land,
              levels=np.linspace(.8, 4.8, 6), linewidths=.5, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].text(tx, ty, titles[i], **text_kw1)
    ax[i].text(map_bnds[0]+0.5, ty+0.5, letters[i], **text_kw2)
    # ONE red outline for the whole NAM domain. nam_domain_outline() unions the south and
    # north sub-domains and dissolves the edge they share, so no internal division is drawn.
    ax[i].add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}
        gl.ylabel_style = {'size': 11} 
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}

cbar_ax1 = fig.add_axes([0.055, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'$\Delta$[‰]', size=12, labelpad=5)
cbar1.ax.tick_params(labelsize=12)

cbar_ax2 = fig.add_axes([0.54, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[0,20,40,60,80], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[%]', size=12, labelpad=5)
cbar2.ax.tick_params(labelsize=12)

# save output
plt.savefig(f'{opath}/modern_climo.png', dpi=1200, bbox_inches='tight')